<a href="https://colab.research.google.com/github/MohsenBahaj/cat-detector/blob/main/DecisionTreeClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/Students Social Media Addiction.csv')

# Drop identifier column
df.drop("Student_ID", axis=1, inplace=True)

# Encode target: 'Yes' -> 1, 'No' -> 0
df['Affects_Academic_Performance'] = df['Affects_Academic_Performance'].map({'No': 0, 'Yes': 1})

# Split features and labels
X = df.drop("Affects_Academic_Performance", axis=1)
y = df["Affects_Academic_Performance"]

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline: OneHotEncode categorical, pass numeric as is
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# Define pipeline with Decision Tree Classifier
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=42))
])

# Split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train the pipeline
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

# Evaluate results
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Decision Tree Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        91

    accuracy                           1.00       141
   macro avg       1.00      1.00      1.00       141
weighted avg       1.00      1.00      1.00       141



In [29]:
# New student sample — note 'Facebook' capitalized if that's how it appears in training data
new_student = pd.DataFrame([{
    'Age': 20,
    'Gender': 'Female',
    'Academic_Level': 'Undergraduate',
    'Country': 'USA',
    'Avg_Daily_Usage_Hours': 1.5,
    'Most_Used_Platform': 'Facebook',  # Capital F
    'Sleep_Hours_Per_Night': 4.0,
    'Mental_Health_Score': 10,
    'Relationship_Status': 'Single',
    'Conflicts_Over_Social_Media': 2,
    'Addicted_Score': 5
}])

# Predict class (0 = No, 1 = Yes)
prediction = clf.predict(new_student)[0]

# Predict probability
prob = clf.predict_proba(new_student)[0]

# Show result
print(f"Prediction: {'Affects Academic Performance' if prediction == 1 else 'Does NOT Affect Performance'}")
print(f"Probability (No): {prob[0]:.2f}, Probability (Yes): {prob[1]:.2f}")


Prediction: Does NOT Affect Performance
Probability (No): 0.70, Probability (Yes): 0.30
